# Airflow Lens — observe and control the pipeline

A control panel for the local Airflow 3.3 instance, driven entirely through its REST API.

**What this can do:** list DAGs and their state, pause and unpause, trigger a run with
configuration, inspect runs and task instances, read task logs, clear failed tasks so they
rerun, read and write Variables and pool slots, and check scheduler health.

**What it deliberately cannot do:** edit DAG source. That lives in `dags/` under git. The
tuning that operators actually need — how many days of telemetry to load, how many threads,
batch sizes, schedules — is exposed through Airflow **Variables**, which the DAGs read. So you
change behaviour here without editing code, and code changes stay reviewable.

> Start Airflow first: `make airflow` (api-server on :8080, scheduler, triggerer).

In [ ]:
from aimternet.observability.airflow_client import AirflowLens, AirflowApiError
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 60)

lens = AirflowLens()
print("Airflow :", lens.version())

## 1. Is Airflow healthy?

`make airflow` runs everything. If you started only the api-server, the scheduler and
triggerer will read `unhealthy` here — that is accurate, not a bug: nothing will actually run.

In [ ]:
health = lens.health()
pd.DataFrame([
    {"component": name, "status": info.get("status"), "last_heartbeat": info.get("latest_scheduler_heartbeat")}
    for name, info in health.items()
])

## 2. The DAGs

In [ ]:
dags = lens.list_dags()          # only_ours=True hides Airflow's bundled examples
if not dags:
    print("No project DAGs registered yet. They land in Phase 11 under dags/.")
    print(f"(Airflow currently knows {len(lens.list_dags(only_ours=False))} example DAGs.)")
else:
    display(pd.DataFrame(dags)[["dag_id", "is_paused", "timetable_summary", "last_parsed_time"]])

## 3. Control: pause, unpause, trigger

The six pipeline DAGs. `bootstrap_raw_landing` is manual-trigger only by design — it is the
one-time load from the EC2 landing directory.

In [ ]:
PIPELINE_DAGS = [
    "bootstrap_raw_landing",        # manual, once
    "rds_to_s3_incremental",
    "dynamodb_to_s3_incremental",
    "curate_silver_gold",
    "load_redshift",
    "reconcile_data",
]

def safe(action, *args, **kwargs):
    """Run a lens call and show the error instead of a traceback if the DAG is absent."""
    try:
        return action(*args, **kwargs)
    except AirflowApiError as exc:
        print(f"  {type(exc).__name__}: {str(exc)[:160]}")
        return None

for dag_id in PIPELINE_DAGS:
    info = safe(lens.get_dag, dag_id)
    if info:
        print(f"  {dag_id:30s} paused={info['is_paused']}  schedule={info.get('timetable_summary')}")

In [ ]:
# Unpause a DAG so the scheduler will run it.
# safe(lens.unpause, "curate_silver_gold")

# Pause it again.
# safe(lens.pause, "curate_silver_gold")

# Trigger a run, optionally overriding behaviour for this run only.
# run = safe(lens.trigger, "bootstrap_raw_landing", conf={"telemetry_days": 7}, note="smoke test")
# run

## 4. Watch a run

Task instances tell you where a run actually is. `state` is the column that matters.

In [ ]:
DAG_TO_WATCH = "bootstrap_raw_landing"

runs = safe(lens.list_runs, DAG_TO_WATCH, limit=5) or []
if runs:
    display(pd.DataFrame(runs)[["dag_run_id", "state", "run_type", "start_date", "end_date"]])
else:
    print(f"No runs of {DAG_TO_WATCH} yet.")

In [ ]:
if runs:
    latest = runs[0]["dag_run_id"]
    tasks = lens.list_task_instances(DAG_TO_WATCH, latest)
    display(pd.DataFrame(tasks)[["task_id", "state", "try_number", "duration", "start_date"]])

## 5. Diagnose and retry

Read the log of a failed task, then clear it. Clearing marks the task (and everything
downstream) for rerun — the normal fix for a transient S3 or network failure.

In [ ]:
# log = lens.task_log(DAG_TO_WATCH, latest, "upload_bronze", try_number=1)
# print(log[-4000:])

In [ ]:
# lens.clear_task(DAG_TO_WATCH, latest, ["upload_bronze"])   # reruns it and its downstream

## 6. Tuning knobs — Variables

These are the values the DAGs read at run time. Changing one here changes pipeline behaviour
on the next run, with no code edit and no redeploy.

| Variable | Effect |
|---|---|
| `aimternet_telemetry_days` | how many of the 62 days of telemetry the loaders ingest |
| `aimternet_load_threads` | thread-pool width for S3 upload and DynamoDB writes |
| `aimternet_batch_size` | rows per batch for the database loaders |
| `aimternet_orphan_member_policy` | `synthesize_stub` or `quarantine` for the D2 cohort |

In [ ]:
DEFAULTS = {
    "aimternet_telemetry_days":       ("62", "Days of telemetry the loaders ingest (max 62)"),
    "aimternet_load_threads":         ("8",  "Thread-pool width for S3 and DynamoDB writes"),
    "aimternet_batch_size":           ("1000", "Rows per batch for database loaders"),
    "aimternet_orphan_member_policy": ("synthesize_stub", "D2 policy: synthesize_stub | quarantine"),
}

for key, (value, description) in DEFAULTS.items():
    try:
        lens.get_variable(key)
    except AirflowApiError:
        lens.set_variable(key, value, description)

pd.DataFrame(lens.list_variables())[["key", "value", "description"]]

In [ ]:
# Change one, and the next DAG run picks it up.
# lens.set_variable("aimternet_telemetry_days", "7", "Days of telemetry the loaders ingest (max 62)")
# lens.get_variable("aimternet_telemetry_days")

## 7. Concurrency — pools

Pools cap how many tasks run at once. The telemetry load is the one worth throttling: it
writes millions of items and will happily saturate the instance.

In [ ]:
pd.DataFrame(lens.list_pools())[["name", "slots", "occupied_slots", "running_slots", "queued_slots"]]

In [ ]:
# Widen or narrow concurrency without touching a DAG file.
# lens.set_pool_slots("default_pool", 64)

## 8. Read a DAG's source

Read-only on purpose. To change a DAG, edit it in `dags/` and commit — the scheduler picks up
the new version on its next parse.

In [ ]:
# print(lens.dag_source("bootstrap_raw_landing"))